# Lesson 3: Prepare next-token training examples

A language model learns by predicting the next token. This lesson prepares those examples: reserve part of the corpus for validation, split one sequence into context and next-token targets, and sample several windows as a mini-batch. The original notebook covers these ideas across cells 7–11; here they are grouped into three steps after a small setup cell.

The matching [C entry point](../c/src/batches_main.c) and [Java entry point](../java/src/main/java/learning/karpathy/gpt1/NextTokenBatchLesson.java) use ordinary arrays. Python introduces a PyTorch tensor because slicing, stacking, and batch shapes are central to the next part of the tutorial.

## Prepare the encoded corpus

This setup repeats the earlier lessons so this notebook runs independently. It reads the same file, rebuilds the character-to-ID mapping, and stores the encoded corpus in a one-dimensional tensor of integer token IDs. `torch.long` is the integer type used for token indices in the reference code.

In [ ]:
from pathlib import Path
import torch

data_candidates = (
    Path.cwd() / "data" / "input.txt",
    Path.cwd() / "gpt-1" / "data" / "input.txt",
    Path.cwd().parent / "data" / "input.txt",
)
dataset_path = next((path.resolve() for path in data_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find gpt-1/data/input.txt. Open the repo root, gpt-1/, or gpt-1/python/.")

text = dataset_path.read_text(encoding="utf-8")
chars = sorted(set(text))
stoi = {character: token_id for token_id, character in enumerate(chars)}
itos = {token_id: character for token_id, character in enumerate(chars)}
encode = lambda sample: [stoi[character] for character in sample]
decode = lambda token_ids: "".join(itos[token_id] for token_id in token_ids)

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Dataset: {dataset_path}")
print(f"Encoded corpus: shape={tuple(data.shape)}, dtype={data.dtype}")
print(f"First 40 token IDs: {data[:40].tolist()}")

## Step 1: Split training and validation data

Keep the first 90% for training and the final 10% for validation. We preserve order instead of shuffling the text, so validation comes from a later part of the corpus. During training, the model will learn from the first part; validation will help us check how well it predicts text it was not trained on. A tensor slice gives us each region without rewriting the token IDs.

In [ ]:
split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]

print(f"All tokens: {len(data):,}")
print(f"Training tokens (first 90%): {len(train_data):,}")
print(f"Validation tokens (last 10%): {len(val_data):,}")

## Step 2: Shift context into next-token targets

With `block_size = 8`, take nine consecutive tokens. The first eight (`x`) are the inputs; the same sequence shifted left by one (`y`) gives the desired next token at each position. The model will eventually predict all eight targets at once, while this loop shows the learning task one position at a time. See reference cells 9–10.

In [ ]:
block_size = 8
window = train_data[:block_size + 1]
x = window[:block_size]
y = window[1:block_size + 1]

print(f"Window IDs: {window.tolist()}")
print(f"x input IDs: {x.tolist()}")
print(f"y target IDs: {y.tolist()}")
print()
for time in range(block_size):
    context = x[:time + 1].tolist()
    target_id = y[time].item()
    print(f"context {context} ({decode(context)!r}) -> target {target_id} ({decode([target_id])!r})")

## Step 3: Sample a mini-batch

A batch contains several independent windows so the model can process them together. `x_batch` and `y_batch` each have shape `(batch_size, block_size)`, here `(4, 8)`. For every row, each target is the next token after the input at that position. `torch.stack` combines one-dimensional windows into a two-dimensional tensor; C and Java show the same row-and-column structure with nested loops and arrays.

The random seed makes Python's sample repeatable. C, Java, and PyTorch use different random-number generators, so their selected windows can differ while obeying the same rule.

In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    source = train_data if split == "train" else val_data
    # The last valid start still needs block_size + 1 source tokens for y.
    starts = torch.randint(len(source) - block_size, (batch_size,)).tolist()
    x_batch = torch.stack([source[start:start + block_size] for start in starts])
    y_batch = torch.stack([source[start + 1:start + block_size + 1] for start in starts])
    return x_batch, y_batch

x_batch, y_batch = get_batch("train")
x_val_batch, y_val_batch = get_batch("val")
print(f"x shape: {tuple(x_batch.shape)}")
print(f"y shape: {tuple(y_batch.shape)}")
print("x inputs:")
print(x_batch)
print("y next-token targets:")
print(y_batch)
print("validation x inputs:")
print(x_val_batch)
print("validation y next-token targets:")
print(y_val_batch)
print(f"validation x/y shapes: {tuple(x_val_batch.shape)}, {tuple(y_val_batch.shape)}")
assert x_batch.shape == y_batch.shape == (batch_size, block_size)
assert x_val_batch.shape == y_val_batch.shape == (batch_size, block_size)
assert torch.equal(x_batch[:, 1:], y_batch[:, :-1])

## Compare and experiment

Run C or Java with **Terminal → Run Task → Run tutorial lesson**, choose lesson `3`, then a step. The [C entry point](../c/src/batches_main.c) stores each batch as a flat row-major array; the [Java entry point](../java/src/main/java/learning/karpathy/gpt1/NextTokenBatchLesson.java) uses `int[][]`. In Java, Ctrl+click `System.arraycopy` or `Random.nextInt` to inspect the standard-library implementation and documentation.

Try changing `block_size` to 4, then 12. How do the shapes and the amount of context change? Why must the target sequence be shifted by exactly one token?

References: [PyTorch tensor indexing and slicing](https://docs.pytorch.org/docs/stable/tensors.html), [PyTorch `torch.stack`](https://docs.pytorch.org/docs/stable/generated/torch.stack.html), and the original [GPT tutorial notebook](../gpt_dev.ipynb), cells 7–11.